In [1]:
from pyspark.sql import SparkSession

spark = SparkSession. \
builder. \
appName("window_practice"). \
config("spark.sql.warehouse.dir", f"/user/itv027484/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [2]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql import Window

In [3]:

from decimal import Decimal

# Initialize Spark session (automatically handled in Databricks)
data = [
    (101, "East", "Electronics", "Laptop", "2026-01-05", 1200.00),
    (102, "East", "Electronics", "Smartphone", "2026-01-10", 800.00),
    (103, "East", "Electronics", "Tablet", "2026-01-15", 500.00),
    (104, "East", "Furniture", "Desk", "2026-01-18", 300.00),
    (105, "East", "Furniture", "Chair", "2026-02-01", 150.00),
    (106, "West", "Electronics", "Laptop", "2026-01-08", 1200.00),
    (107, "West", "Electronics", "Smartphone", "2026-01-12", 850.00),
    (108, "West", "Electronics", "Monitor", "2026-01-20", 300.00),
    (109, "West", "Furniture", "Desk", "2026-02-05", 350.00),
    (110, "West", "Furniture", "Couch", "2026-02-15", 900.00),
    (111, "North", "Electronics", "Laptop", "2026-01-07", 1200.00),
    (112, "North", "Electronics", "Tablet", "2026-01-14", 500.00),
    (113, "North", "Furniture", "Chair", "2026-01-25", 150.00),
    (114, "North", "Furniture", "Desk", "2026-02-10", 300.00),
    (115, "South", "Electronics", "Smartphone", "2026-01-11", 800.00),
    (116, "South", "Electronics", "Monitor", "2026-01-19", 300.00),
    (117, "South", "Furniture", "Couch", "2026-02-02", 900.00),
    (118, "South", "Furniture", "Chair", "2026-02-12", 150.00),
]


In [4]:

columns = ["sale_id", "region", "category", "product", "sale_date", "amount"]

# Create DataFrame
df = spark.createDataFrame(data, columns)
df = df.withColumn("sale_date", to_date(col("sale_date")))

# Register as Temp View for Spark SQL
df.createOrReplaceTempView("sales_data")


In [5]:
# Question 1 (Ranking Comparison):
# Within each category, rank the sales by amount in descending order. 
# Add three separate columns to the output using row_number(), rank(), and dense_rank(). 
# Pay close attention to how tie amounts (e.g., $1200 for Laptops or $150 for Chairs) are handled across the three functions.

In [6]:
df.show(3)

+-------+------+-----------+----------+----------+------+
|sale_id|region|   category|   product| sale_date|amount|
+-------+------+-----------+----------+----------+------+
|    101|  East|Electronics|    Laptop|2026-01-05|1200.0|
|    102|  East|Electronics|Smartphone|2026-01-10| 800.0|
|    103|  East|Electronics|    Tablet|2026-01-15| 500.0|
+-------+------+-----------+----------+----------+------+
only showing top 3 rows



In [7]:
catg = Window.partitionBy("category").orderBy(desc("amount"))

In [8]:
df1 = df. \
withColumn("sales_rank", rank().over(catg)). \
withColumn("sales_rownum", row_number().over(catg)). \
withColumn("sales_denserank", dense_rank().over(catg))

In [9]:
df1.show()

+-------+------+-----------+----------+----------+------+----------+------------+---------------+
|sale_id|region|   category|   product| sale_date|amount|sales_rank|sales_rownum|sales_denserank|
+-------+------+-----------+----------+----------+------+----------+------------+---------------+
|    101|  East|Electronics|    Laptop|2026-01-05|1200.0|         1|           1|              1|
|    106|  West|Electronics|    Laptop|2026-01-08|1200.0|         1|           2|              1|
|    111| North|Electronics|    Laptop|2026-01-07|1200.0|         1|           3|              1|
|    107|  West|Electronics|Smartphone|2026-01-12| 850.0|         4|           4|              2|
|    102|  East|Electronics|Smartphone|2026-01-10| 800.0|         5|           5|              3|
|    115| South|Electronics|Smartphone|2026-01-11| 800.0|         5|           6|              3|
|    103|  East|Electronics|    Tablet|2026-01-15| 500.0|         7|           7|              4|
|    112| North|Elec

In [10]:
spark.sql("""

select sale_id,region,category,product,sale_date,amount,
rank() over(partition by category order by amount desc) as rank,
row_number() over(partition by category order by amount desc) as row_num,
dense_rank() over(partition by category order by amount desc) as dense_rank
from sales_data
"""
).show()

+-------+------+-----------+----------+----------+------+----+-------+----------+
|sale_id|region|   category|   product| sale_date|amount|rank|row_num|dense_rank|
+-------+------+-----------+----------+----------+------+----+-------+----------+
|    101|  East|Electronics|    Laptop|2026-01-05|1200.0|   1|      1|         1|
|    106|  West|Electronics|    Laptop|2026-01-08|1200.0|   1|      2|         1|
|    111| North|Electronics|    Laptop|2026-01-07|1200.0|   1|      3|         1|
|    107|  West|Electronics|Smartphone|2026-01-12| 850.0|   4|      4|         2|
|    102|  East|Electronics|Smartphone|2026-01-10| 800.0|   5|      5|         3|
|    115| South|Electronics|Smartphone|2026-01-11| 800.0|   5|      6|         3|
|    103|  East|Electronics|    Tablet|2026-01-15| 500.0|   7|      7|         4|
|    112| North|Electronics|    Tablet|2026-01-14| 500.0|   7|      8|         4|
|    108|  West|Electronics|   Monitor|2026-01-20| 300.0|   9|      9|         5|
|    116| South|

In [11]:
# Question 2 (Top N per Group):
# Find the top 2 highest sales transactions (amount) for each region. Filter out any transactions ranked lower than 2 using dense_rank().

In [12]:
region_window = Window.partitionBy("region").orderBy(desc("amount"))

In [13]:
df2 = df.withColumn("sale_rank",dense_rank().over(region_window)).filter("sale_rank < 3")

In [14]:
df2.show()

+-------+------+-----------+----------+----------+------+---------+
|sale_id|region|   category|   product| sale_date|amount|sale_rank|
+-------+------+-----------+----------+----------+------+---------+
|    117| South|  Furniture|     Couch|2026-02-02| 900.0|        1|
|    115| South|Electronics|Smartphone|2026-01-11| 800.0|        2|
|    101|  East|Electronics|    Laptop|2026-01-05|1200.0|        1|
|    102|  East|Electronics|Smartphone|2026-01-10| 800.0|        2|
|    106|  West|Electronics|    Laptop|2026-01-08|1200.0|        1|
|    110|  West|  Furniture|     Couch|2026-02-15| 900.0|        2|
|    111| North|Electronics|    Laptop|2026-01-07|1200.0|        1|
|    112| North|Electronics|    Tablet|2026-01-14| 500.0|        2|
+-------+------+-----------+----------+----------+------+---------+



In [15]:
spark.sql("""
select  * from (
select sale_id,region,category,product,sale_date,amount,
dense_rank() over(partition by region order by amount desc) as dense_rank
from sales_data ) a
where a.dense_rank < 3
"""
).show()

+-------+------+-----------+----------+----------+------+----------+
|sale_id|region|   category|   product| sale_date|amount|dense_rank|
+-------+------+-----------+----------+----------+------+----------+
|    117| South|  Furniture|     Couch|2026-02-02| 900.0|         1|
|    115| South|Electronics|Smartphone|2026-01-11| 800.0|         2|
|    101|  East|Electronics|    Laptop|2026-01-05|1200.0|         1|
|    102|  East|Electronics|Smartphone|2026-01-10| 800.0|         2|
|    106|  West|Electronics|    Laptop|2026-01-08|1200.0|         1|
|    110|  West|  Furniture|     Couch|2026-02-15| 900.0|         2|
|    111| North|Electronics|    Laptop|2026-01-07|1200.0|         1|
|    112| North|Electronics|    Tablet|2026-01-14| 500.0|         2|
+-------+------+-----------+----------+----------+------+----------+



In [16]:
# Question 3 (Running Total & Partitioning):
# For each region, calculate a cumulative running total of sales amount ordered by sale_date. 
# Include the product, sale_date, amount, and the running_total column.

In [17]:
window_tot = Window.partitionBy("region").orderBy("sale_date").rowsBetween(Window.unboundedPreceding,Window.currentRow)

In [18]:
df3 = df.withColumn('run_tot',sum("amount").over(window_tot))

In [19]:
df3.show()

+-------+------+-----------+----------+----------+------+-------+
|sale_id|region|   category|   product| sale_date|amount|run_tot|
+-------+------+-----------+----------+----------+------+-------+
|    115| South|Electronics|Smartphone|2026-01-11| 800.0|  800.0|
|    116| South|Electronics|   Monitor|2026-01-19| 300.0| 1100.0|
|    117| South|  Furniture|     Couch|2026-02-02| 900.0| 2000.0|
|    118| South|  Furniture|     Chair|2026-02-12| 150.0| 2150.0|
|    101|  East|Electronics|    Laptop|2026-01-05|1200.0| 1200.0|
|    102|  East|Electronics|Smartphone|2026-01-10| 800.0| 2000.0|
|    103|  East|Electronics|    Tablet|2026-01-15| 500.0| 2500.0|
|    104|  East|  Furniture|      Desk|2026-01-18| 300.0| 2800.0|
|    105|  East|  Furniture|     Chair|2026-02-01| 150.0| 2950.0|
|    106|  West|Electronics|    Laptop|2026-01-08|1200.0| 1200.0|
|    107|  West|Electronics|Smartphone|2026-01-12| 850.0| 2050.0|
|    108|  West|Electronics|   Monitor|2026-01-20| 300.0| 2350.0|
|    109| 

In [20]:
spark.sql("""

select sale_id,region,category,product,sale_date,amount,
sum(amount) over(partition by region order by sale_date  rows between unbounded Preceding and current row ) as run_tot
from sales_data
"""
).show()

+-------+------+-----------+----------+----------+------+-------+
|sale_id|region|   category|   product| sale_date|amount|run_tot|
+-------+------+-----------+----------+----------+------+-------+
|    115| South|Electronics|Smartphone|2026-01-11| 800.0|  800.0|
|    116| South|Electronics|   Monitor|2026-01-19| 300.0| 1100.0|
|    117| South|  Furniture|     Couch|2026-02-02| 900.0| 2000.0|
|    118| South|  Furniture|     Chair|2026-02-12| 150.0| 2150.0|
|    101|  East|Electronics|    Laptop|2026-01-05|1200.0| 1200.0|
|    102|  East|Electronics|Smartphone|2026-01-10| 800.0| 2000.0|
|    103|  East|Electronics|    Tablet|2026-01-15| 500.0| 2500.0|
|    104|  East|  Furniture|      Desk|2026-01-18| 300.0| 2800.0|
|    105|  East|  Furniture|     Chair|2026-02-01| 150.0| 2950.0|
|    106|  West|Electronics|    Laptop|2026-01-08|1200.0| 1200.0|
|    107|  West|Electronics|Smartphone|2026-01-12| 850.0| 2050.0|
|    108|  West|Electronics|   Monitor|2026-01-20| 300.0| 2350.0|
|    109| 

In [21]:
# Question 4 (Period-over-Period Delta):
# Within each region, sort records by sale_date and 
# find the difference in sales amount between the current transaction and 
# the previous transaction in that same region using lag().

In [22]:
lag_window = Window.partitionBy("region").orderBy("sale_date")

In [23]:
df4 = df.withColumn('prev_sales', lag("amount").over(lag_window))

In [24]:
df4.withColumn('sales_diff',expr("amount-prev_sales")).show()

+-------+------+-----------+----------+----------+------+----------+----------+
|sale_id|region|   category|   product| sale_date|amount|prev_sales|sales_diff|
+-------+------+-----------+----------+----------+------+----------+----------+
|    115| South|Electronics|Smartphone|2026-01-11| 800.0|      null|      null|
|    116| South|Electronics|   Monitor|2026-01-19| 300.0|     800.0|    -500.0|
|    117| South|  Furniture|     Couch|2026-02-02| 900.0|     300.0|     600.0|
|    118| South|  Furniture|     Chair|2026-02-12| 150.0|     900.0|    -750.0|
|    101|  East|Electronics|    Laptop|2026-01-05|1200.0|      null|      null|
|    102|  East|Electronics|Smartphone|2026-01-10| 800.0|    1200.0|    -400.0|
|    103|  East|Electronics|    Tablet|2026-01-15| 500.0|     800.0|    -300.0|
|    104|  East|  Furniture|      Desk|2026-01-18| 300.0|     500.0|    -200.0|
|    105|  East|  Furniture|     Chair|2026-02-01| 150.0|     300.0|    -150.0|
|    106|  West|Electronics|    Laptop|2

In [25]:
spark.sql("""
select sale_id,region,category,product,sale_date,amount,
lag(amount) over(partition by region order by sale_date) as prev_sales,
amount - lag(amount) over(partition by region order by sale_date) as sales_diff
from sales_data
"""
).show()

+-------+------+-----------+----------+----------+------+----------+----------+
|sale_id|region|   category|   product| sale_date|amount|prev_sales|sales_diff|
+-------+------+-----------+----------+----------+------+----------+----------+
|    115| South|Electronics|Smartphone|2026-01-11| 800.0|      null|      null|
|    116| South|Electronics|   Monitor|2026-01-19| 300.0|     800.0|    -500.0|
|    117| South|  Furniture|     Couch|2026-02-02| 900.0|     300.0|     600.0|
|    118| South|  Furniture|     Chair|2026-02-12| 150.0|     900.0|    -750.0|
|    101|  East|Electronics|    Laptop|2026-01-05|1200.0|      null|      null|
|    102|  East|Electronics|Smartphone|2026-01-10| 800.0|    1200.0|    -400.0|
|    103|  East|Electronics|    Tablet|2026-01-15| 500.0|     800.0|    -300.0|
|    104|  East|  Furniture|      Desk|2026-01-18| 300.0|     500.0|    -200.0|
|    105|  East|  Furniture|     Chair|2026-02-01| 150.0|     300.0|    -150.0|
|    106|  West|Electronics|    Laptop|2

In [26]:
# Question 5 (Basic Pivot):
# Pivot the dataset to aggregate total sales amount where rows represent category and columns represent each region (East, West, North, South).

In [27]:
df4 = df.groupBy("category").pivot("region", ["East", "West", "North", "South"]).sum("amount")

In [28]:
df4.show()

+-----------+------+------+------+------+
|   category|  East|  West| North| South|
+-----------+------+------+------+------+
|Electronics|2500.0|2350.0|1700.0|1100.0|
|  Furniture| 450.0|1250.0| 450.0|1050.0|
+-----------+------+------+------+------+



In [29]:
spark.sql("""
select * from (
select category,region,amount from sales_data
)pivot( sum(amount) for region in('East', 'West', 'North', 'South') 
      ) 
"""
).show()

+-----------+------+------+------+------+
|   category|  East|  West| North| South|
+-----------+------+------+------+------+
|Electronics|2500.0|2350.0|1700.0|1100.0|
|  Furniture| 450.0|1250.0| 450.0|1050.0|
+-----------+------+------+------+------+



In [30]:
# Question 6 (Advanced Pivot with Null Replacement):
# Pivot the dataset by sale_date month (extract month from sale_date) as rows and category as columns, showing the sum of amount. Fill any resulting null values with 0.

In [32]:
df5 = df.withColumn("sale_month",month("sale_date")).groupBy("sale_month").pivot("category").agg(sum("amount")).na.fill(0)

In [33]:
df5.show()

+----------+-----------+---------+
|sale_month|Electronics|Furniture|
+----------+-----------+---------+
|         1|     7650.0|    450.0|
|         2|        0.0|   2750.0|
+----------+-----------+---------+



In [38]:
spark.sql("""

select  
sale_month,
coalesce(Electronics,0) as Electronics,
coalesce(Furniture,0) as Furniture
from 
(
select  month(sale_date) as sale_month, category,amount from sales_data
)
  pivot( sum(amount) for category in('Electronics', 'Furniture') ) 
  
"""
).show()

+----------+-----------+---------+
|sale_month|Electronics|Furniture|
+----------+-----------+---------+
|         1|     7650.0|    450.0|
|         2|        0.0|   2750.0|
+----------+-----------+---------+

